# 01 - Data audit

Purpose: inspect the open datasets before writing production ingestion code. This notebook should answer: what files do we need, what columns do they contain, what geography do they use, and what licence/attribution obligations follow them?

Stage 0 rule: capture findings here, then move stable logic into `scripts/` and `app/`.

## What this notebook does

This is the first quality gate for the project. Before building routing or optimisation logic, it checks whether the source data is identifiable, reproducible, legally usable, and joinable at the right geography.

The outputs should help answer four practical questions:

- Which datasets are required for Milestone 1?
- Where did each dataset come from, and what licence applies?
- Which local files have actually been downloaded?
- Do the files expose the columns needed for LSOA joins, population weighting, deprivation scoring, and stop filtering?

## Source register

The deprivation source is IMD 2025, not IMD 2019. Use updated v2 workbooks where GOV.UK marks them as corrected.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"

DATA_RAW, DATA_PROCESSED

(PosixPath('/Users/dannyyy/Documents/AccessBridge (Phase2)/data/raw'),
 PosixPath('/Users/dannyyy/Documents/AccessBridge (Phase2)/data/processed'))

### Output: project data paths

The output should show absolute paths for `data/raw` and `data/processed`. This confirms the notebook is resolving paths from the project root rather than from a random working directory.

If these paths do not point inside the AccessBridge workspace, restart the notebook from the correct directory before downloading or processing data.

In [2]:
source_register = pd.DataFrame(
    [
        {
            "dataset": "English Indices of Deprivation 2025 / IMD 2025",
            "source_page": "https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025",
            "licence": "Open Government Licence v3.0",
            "role": "LSOA deprivation ranks, scores, and deciles",
            "expected_local_file": "data/raw/imd_2025_file_1_index_of_multiple_deprivation.xlsx",
            "stage0_check": "Confirm LSOA code/name columns, IMD rank, IMD score, and IMD decile.",
        },
        {
            "dataset": "ONS LSOA boundaries",
            "source_page": "https://geoportal.statistics.gov.uk/",
            "licence": "Open Government Licence v3.0",
            "role": "LSOA geometry for choropleth and population-weighted centroids",
            "expected_local_file": "data/raw/lsoa_boundaries.*",
            "stage0_check": "Confirm geography year matches IMD/population joins and CRS is known.",
        },
        {
            "dataset": "ONS population estimates",
            "source_page": "https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/lowersuperoutputareamidyearpopulationestimatesnationalstatistics",
            "licence": "Open Government Licence v3.0",
            "role": "Population weights by small area",
            "expected_local_file": "data/raw/ons_lsoa_population_broad_age_mid_2022_revised_to_mid_2024.xlsx",
            "stage0_check": "Confirm mid-2024 total population field, broad age groups, and Census 2021 LSOA geography.",
        },
        {
            "dataset": "NaPTAN",
            "source_page": "https://beta-naptan.dft.gov.uk/download",
            "licence": "Open Government Licence v3.0",
            "role": "Existing public-transport access nodes and interchange candidates",
            "expected_local_file": "data/raw/naptan_stops.csv",
            "stage0_check": "Filter Birmingham-area stops and identify bus, tram, rail, and interchange nodes.",
        },
        {
            "dataset": "GTFS transit feed",
            "source_page": "https://www.bus-data.dft.gov.uk/",
            "licence": "BODS/open transit feed terms; attribute source",
            "role": "Routes, stops, trips, and timetables for R5",
            "expected_local_file": "data/raw/gtfs.zip",
            "stage0_check": "Record operator/feed, download date, service date range, and Birmingham coverage.",
        },
        {
            "dataset": "OpenStreetMap",
            "source_page": "https://www.openstreetmap.org/",
            "licence": "Open Database Licence (ODbL)",
            "role": "Street network for walking access and R5 network build",
            "expected_local_file": "data/raw/osm_extract.*",
            "stage0_check": "Record extract date and avoid redistributing derived OSM databases without ODbL compliance.",
        },
    ]
)

source_register

,dataset,source_page,licence,role,expected_local_file,stage0_check
0,English Indices of Deprivation 2025 / IMD 2025,https://www.gov.uk/government/statistics/engli...,Open Government Licence v3.0,"LSOA deprivation ranks, scores, and deciles",data/raw/imd_2025_file_1_index_of_multiple_dep...,"Confirm LSOA code/name columns, IMD rank, IMD ..."
1,ONS LSOA boundaries,https://geoportal.statistics.gov.uk/,Open Government Licence v3.0,LSOA geometry for choropleth and population-we...,data/raw/lsoa_boundaries.*,Confirm geography year matches IMD/population ...
2,ONS population estimates,https://www.ons.gov.uk/peoplepopulationandcomm...,Open Government Licence v3.0,Population weights by small area,data/raw/ons_lsoa_population_broad_age_mid_202...,"Confirm mid-2024 total population field, broad..."
3,NaPTAN,https://beta-naptan.dft.gov.uk/download,Open Government Licence v3.0,Existing public-transport access nodes and int...,data/raw/naptan_stops.csv,"Filter Birmingham-area stops and identify bus,..."
4,GTFS transit feed,https://www.bus-data.dft.gov.uk/,BODS/open transit feed terms; attribute source,"Routes, stops, trips, and timetables for R5",data/raw/gtfs.zip,"Record operator/feed, download date, service d..."
5,OpenStreetMap,https://www.openstreetmap.org/,Open Database Licence (ODbL),Street network for walking access and R5 netwo...,data/raw/osm_extract.*,Record extract date and avoid redistributing d...


### Output: source register table

This table is the data contract for Stage 0. Each row is one dataset the project expects to use, with its source page, licence, role in the pipeline, expected local filename, and the specific audit check to perform.

A good result here is not about the files existing yet. It is about making the dependency list explicit before writing fetch scripts. Later, `scripts/fetch_data.py` should be able to recreate the files listed in `expected_local_file` or document why a different filename is used.

Pay special attention to the licence column: the app and README must carry OGL, OSM/ODbL, and transit-feed attribution.

## Fetch status

If you ran this notebook before downloading data, the next tables will be empty or marked `False`. That is not a bug: raw data files are gitignored and must be fetched locally.

For the first real audit pass, run this command from the project root:

```bash
python3 scripts/fetch_data.py
```

That downloads the light Stage 0 files: IMD 2025 File 1, IMD 2025 File 7, and ONS LSOA broad-age population estimates. Heavier geospatial/network data comes later.

In [3]:
def expected_file_status(register: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in register.to_dict(orient="records"):
        expected = row["expected_local_file"]
        if "*" in expected:
            matches = sorted(ROOT.glob(expected))
            exists = bool(matches)
            matched_files = [path.relative_to(ROOT).as_posix() for path in matches]
            size_mb = sum(path.stat().st_size for path in matches) / 1_000_000
        else:
            path = ROOT / expected
            exists = path.exists()
            matched_files = [expected] if exists else []
            size_mb = path.stat().st_size / 1_000_000 if exists else 0

        rows.append(
            {
                "dataset": row["dataset"],
                "expected_local_file": expected,
                "exists": exists,
                "matched_files": matched_files,
                "size_mb": round(size_mb, 3),
            }
        )
    return pd.DataFrame(rows, columns=["relative_path", "size_mb"])


expected_file_status(source_register)

,relative_path,size_mb
0,NaN,1.469
1,NaN,0.000
2,NaN,13.063
3,NaN,0.000
4,NaN,0.000
5,NaN,0.000


### Output: expected-file status table

This table is the first thing to check after running the notebook. `exists = False` means the dataset has not been downloaded into `data/raw` yet.

For the first notebook, it is enough if the IMD and ONS population rows become `True` after running `python3 scripts/fetch_data.py`. The boundary, NaPTAN, GTFS, and OSM rows are expected to remain missing until the later Stage 0 notebooks need them.

## Local file inventory

Run this after placing downloaded files in `data/raw/`. Keep the inventory in the notebook output for reproducibility.

In [4]:
def list_files(path: Path) -> pd.DataFrame:
    rows = []
    for file_path in sorted(path.glob("**/*")):
        if file_path.is_file() and file_path.name != ".gitkeep":
            rows.append(
                {
                    "relative_path": file_path.relative_to(ROOT).as_posix(),
                    "size_mb": round(file_path.stat().st_size / 1_000_000, 3),
                }
            )
    return pd.DataFrame(rows)

list_files(DATA_RAW)

,relative_path,size_mb
0,data/raw/imd_2025_file_1_index_of_multiple_dep...,1.469
1,data/raw/imd_2025_file_7_all_ranks_scores_deci...,9.898
2,data/raw/ons_lsoa_population_broad_age_mid_202...,13.063
3,data/raw/provenance_stage0.json,0.003


### Output: local raw-file inventory

This output lists every real file currently present under `data/raw`, excluding `.gitkeep`. At the start of the project it may be empty; after data download it should show the IMD workbook, boundary file, population file, NaPTAN file, GTFS archive, and any OSM extract or metadata file.

Use this as a reproducibility check. If a notebook result depends on a file that is not listed here, the workflow is relying on hidden local state and should be fixed.

## Tabular audit helper

Use this for CSVs once files exist. For Excel workbooks, inspect sheet names first and then load the relevant sheet explicitly.

In [5]:
def audit_csv(path: Path, sample_rows: int = 5) -> dict[str, object]:
    frame = pd.read_csv(path, nrows=sample_rows)
    return {
        "path": path.relative_to(ROOT).as_posix(),
        "columns": list(frame.columns),
        "sample": frame.head(sample_rows),
    }

# Example once a CSV exists:
# audit = audit_csv(DATA_RAW / "naptan_stops.csv")
# audit["columns"], audit["sample"]

### Output: CSV audit dictionary

When you uncomment the example for a real CSV, the helper returns three things: the relative path, the column names, and a small sample of rows.

This is where we confirm whether a file has the keys needed by the pipeline. For example, NaPTAN should expose stable stop IDs plus coordinates; population data should expose an LSOA code and a population count; any geospatial boundary file should expose an LSOA code compatible with IMD 2025.

For Excel files like the IMD workbook, inspect sheet names first with `pd.ExcelFile(path).sheet_names`, then load the relevant sheet explicitly. Do not guess column names in production code until this notebook has verified them.

## Real-data smoke checks

Once `scripts/fetch_data.py` has run, this section confirms that the first three light files can be opened. This is not full cleaning yet; it is just the first proof that the files exist and expose inspectable structure.

In [6]:
light_stage0_files = {
    "imd_file1": DATA_RAW / "imd_2025_file_1_index_of_multiple_deprivation.xlsx",
    "imd_file7": DATA_RAW / "imd_2025_file_7_all_ranks_scores_deciles_population_denominators.csv",
    "lsoa_population_broad": DATA_RAW / "ons_lsoa_population_broad_age_mid_2022_revised_to_mid_2024.xlsx",
}

{key: path.exists() for key, path in light_stage0_files.items()}

{'imd_file1': True, 'imd_file7': True, 'lsoa_population_broad': True}

### Output: light-file existence check

All three values should be `True` after the fetch script has run. If any value is `False`, rerun `python3 scripts/fetch_data.py` from the project root.

In [7]:
def excel_sheet_summary(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame([{"file": path.name, "status": "missing", "sheet_names": []}])
    try:
        workbook = pd.ExcelFile(path)
    except ImportError as exc:
        return pd.DataFrame(
            [
                {
                    "file": path.name,
                    "status": f"cannot open - install optional Excel dependency: {exc.name}",
                    "sheet_names": [],
                }
            ]
        )
    return pd.DataFrame(
        [
            {
                "file": path.name,
                "status": "ok",
                "sheet_names": workbook.sheet_names,
            }
        ]
    )


pd.concat(
    [
        excel_sheet_summary(light_stage0_files["imd_file1"]),
        excel_sheet_summary(light_stage0_files["lsoa_population_broad"]),
    ],
    ignore_index=True,
)

,file,status,sheet_names
0,imd_2025_file_1_index_of_multiple_deprivation....,cannot open - install optional Excel dependenc...,[]
1,ons_lsoa_population_broad_age_mid_2022_revised...,cannot open - install optional Excel dependenc...,[]


### Output: Excel sheet summary

This output tells us which sheets are inside the IMD and ONS population workbooks. The next production ingestion step should read explicit sheet names based on this output.

If the status says an optional Excel dependency is missing, install `openpyxl` in the notebook environment before reading `.xlsx` files with pandas.

In [8]:
imd_file7_path = light_stage0_files["imd_file7"]

if imd_file7_path.exists():
    imd_file7_sample = pd.read_csv(imd_file7_path, nrows=5)
    display(imd_file7_sample)
    print("Columns:")
    print(list(imd_file7_sample.columns))
else:
    print(f"Missing {imd_file7_path.relative_to(ROOT)}")

,LSOA code (2021),LSOA name (2021),Local Authority District code (2024),Local Authority District name (2024),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Score,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2022,Dependent Children aged 0-15: mid 2022,Older population aged 60 and over: mid 2022,Working age population 18-66 (for use with Employment Deprivation Domain): mid 2022
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10,...,1.207,1105,1,1.414,1586,1,1795,149,520,1248
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10,...,0.355,9591,3,1.839,592,1,1671,81,387,1324
2,E01000003,City of London 001C,E09000001,City of London,9.250,25913,8,0.107,25167,8,...,0.318,10175,4,1.679,903,1,1896,136,432,1469
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5,...,0.012,15502,5,2.065,303,1,1737,177,160,1448
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3,...,0.399,8934,3,0.400,9136,3,1837,397,225,1260


Columns:
['LSOA code (2021)', 'LSOA name (2021)', 'Local Authority District code (2024)', 'Local Authority District name (2024)', 'Index of Multiple Deprivation (IMD) Score', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)', 'Income Score (rate)', 'Income Rank (where 1 is most deprived)', 'Income Decile (where 1 is most deprived 10% of LSOAs)', 'Employment Score (rate)', 'Employment Rank (where 1 is most deprived)', 'Employment Decile (where 1 is most deprived 10% of LSOAs)', 'Education, Skills and Training Score', 'Education, Skills and Training Rank (where 1 is most deprived)', 'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)', 'Health Deprivation and Disability Score', 'Health Deprivation and Disability Rank (where 1 is most deprived)', 'Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs)', 'Crime Score', 'Crime Rank (w

### Output: IMD File 7 sample and columns

This is the first real look at the deprivation data. We need to confirm that the file includes LSOA identifiers plus IMD rank, score, decile, and population denominator fields.

These column names will drive the cleaning code later, so copy any surprising names or formatting issues into the findings log rather than fixing them silently.

## Findings log

- IMD source decision: use IMD 2025 / English Indices of Deprivation 2025.
- Geography alignment to confirm: IMD 2025 LSOA codes, boundary vintage, and population estimate geography.
- Licence notes to carry into the app footer: OGL attribution, OSM contributors and ODbL, BODS/transit feed attribution.
- Open question: exact GTFS feed/operator scope for Birmingham Knowledge Quarter routing.